### DEMO on ml100k dataset

Trying the algorithm in both version (serial and parallelized), to assert the correctness of the implementation. Test done on the smallest dataset.

In [6]:
import os, sys, time
import numpy as np
import pandas as pd
from src.data_utils import load_ml100k_split
from src.ccdpp import train_ccdpp_fixed, predict, rmse
from src.spark_utils import get_spark_session
from src.ccdpp_spark import build_grouped, train_ccdpp_spark

data = load_ml100k_split("../ml-100k", base_file="u5.base", test_file="u5.test")
train, m, n = data["train"], data["m"], data["n"]
R_train = data["R_train"]
u_test, i_test, y_true = data["u_test"], data["i_test"], data["y_true"]

print(f"Users: {m}, Items: {n}, Rating training samples: {len(train)}, Test samples: {len(y_true)}")

Users: 943, Items: 1650, Rating training samples: 80000, Test samples: 19964


In [7]:
sparsity = 1 - len(train) / (m * n)
print(f"Training data sparsity: {sparsity:.4f}")
train['rating'].value_counts().sort_index()

Training data sparsity: 0.9486


rating
1     4952
2     8993
3    21668
4    27354
5    17033
Name: count, dtype: int64

In [8]:
# Naive baselines
global_mean = train['rating'].mean()
user_mean = train.groupby('user_idx')['rating'].mean()
item_mean = train.groupby('item_idx')['rating'].mean()

pred_global = np.full(len(y_true), global_mean)
pred_user = pd.Series(u_test).map(user_mean).fillna(global_mean).values
pred_item = pd.Series(i_test).map(item_mean).fillna(global_mean).values

print(f"RMSE global mean : {rmse(y_true, pred_global):.4f}")
print(f"RMSE user mean : {rmse(y_true, pred_user):.4f}")
print(f"RMSE item mean : {rmse(y_true, pred_item):.4f}")

RMSE global mean : 1.1180
RMSE user mean : 1.0395
RMSE item mean : 1.0214


Same parameters and number of iterations, used for both versions.

In [ ]:
# best hparams after validation (done in other notebook)
BEST_K, BEST_LAMBDA, BEST_ITER = 2, 1.2, 27

In [10]:
# training serial numpy version
start = time.time()
W_np, H_np = train_ccdpp_fixed(R_train, k=BEST_K, lam=BEST_LAMBDA, n_iter=BEST_ITER)
time_np = time.time() - start

y_pred_np = predict(W_np, H_np, u_test, i_test)
rmse_np = rmse(y_true, y_pred_np)
print(f"CCD++ serial (numpy): RMSE test={rmse_np:.4f}, time={time_np:.2f}s")

CCD++ serial (numpy): RMSE test=0.9276, time=0.17s


In [12]:
# training Spark version
spark = get_spark_session(app_name="ccdpp_ml100k", num_cores=4)
sc = spark.sparkContext

NUM_PARTITIONS = 4

ratings_rdd = sc.parallelize(list(zip(
    train['user_idx'].values, train['item_idx'].values, train['rating'].values
)))
R_rows = build_grouped(ratings_rdd, group_by_user=True, num_partitions=NUM_PARTITIONS)
R_cols = build_grouped(ratings_rdd, group_by_user=False, num_partitions=NUM_PARTITIONS)
R_rows.count(); R_cols.count()

start = time.time()
W_spark, H_spark = train_ccdpp_spark(sc, R_rows, R_cols, m=m, n=n,
                                     k=BEST_K, lam=BEST_LAMBDA, n_iter=BEST_ITER,
                                     checkpoint_every=5)

time_spark = time.time() - start

y_pred_spark = predict(W_spark, H_spark, u_test, i_test)
rmse_spark = rmse(y_true, y_pred_spark)
print(f"CCD++ Spark: RMSE test={rmse_spark:.4f}, time={time_spark:.2f}s")
print(f"Delta RMSE vs numpy: {abs(rmse_spark - rmse_np):.5f}")

CCD++ Spark: RMSE test=0.9276, time=2777.98s
Delta RMSE vs numpy: 0.00000


As expected, since the algorithm is the same, the resulting test RMSE obtained is the same.

The Spark version took much more time, maybe for the overhead of the operations on this small dataset.

In [14]:
# idea of scalability analysis on cores (demo)
core_values = [1, 2, 4]
scal_results = []

for nc in core_values:
    spark.stop()
    spark = get_spark_session(app_name=f"ccdpp_cores_{nc}", num_cores=nc)
    sc = spark.sparkContext

    num_partitions = nc * 3
    ratings_rdd = sc.parallelize(list(zip(
        train['user_idx'].values, train['item_idx'].values, train['rating'].values
    )))
    R_rows = build_grouped(ratings_rdd, group_by_user=True, num_partitions=NUM_PARTITIONS)
    R_cols = build_grouped(ratings_rdd, group_by_user=False, num_partitions=NUM_PARTITIONS)
    R_rows.count(); R_cols.count()

    times = []
    for run in range(3):
        start = time.time()
        W, H = train_ccdpp_spark(sc, R_rows, R_cols, m=m, n=n,
                                 k=BEST_K, lam=BEST_LAMBDA, n_iter=5,
                                 checkpoint_every=10)
        times.append(time.time() - start)

    mean_time = np.mean(times)
    scal_results.append({"num_cores": nc, "num_partitions": num_partitions,
                         "mean_time": mean_time, "std_time": np.std(times)})
    print(f"cores={nc}: mean time={mean_time:.2f}s (3 run)")

df_scal = pd.DataFrame(scal_results)
baseline = df_scal.loc[df_scal["num_cores"]==1, "mean_time"].values[0]
df_scal["speedup"] = baseline / df_scal["mean_time"]
df_scal["efficiency"] = df_scal["speedup"] / df_scal["num_cores"]
df_scal

cores=1: mean time=623.87s (3 run)
cores=2: mean time=533.29s (3 run)
cores=4: mean time=533.31s (3 run)


,num_cores,num_partitions,mean_time,std_time,speedup,efficiency
0,1,3,623.869887,10.072386,1.000000,1.000000
1,2,6,533.293080,15.006899,1.169844,0.584922
2,4,12,533.312639,4.032037,1.169801,0.292450


To investigate: there might be a bottleneck using 4 cores, the time is the same (as 2 cores) and the efficiency is lower. Maybe I should keep the partitions fixed insted of doing num_partitions = nc * 3, where nc is the number of cores.

In [16]:
results_log = [
    {"model": "Global mean", "rmse_test": rmse(y_true, pred_global)},
    {"model": "Usr mean", "rmse_test": rmse(y_true, pred_user)},
    {"model": "Item mean", "rmse_test": rmse(y_true, pred_item)},
    {"model": "CCD++ (numpy)", "rmse_test": rmse_np, "train_time_sec": time_np},
    {"model": "CCD++ (Spark)", "rmse_test": rmse_spark, "train_time_sec": time_spark},
]

df_results = pd.DataFrame(results_log)
# to_csv maybe
df_results

,model,rmse_test,train_time_sec
0,Global mean,1.118008,NaN
1,Usr mean,1.039451,NaN
2,Item mean,1.021422,NaN
3,CCD++ (numpy),0.927611,0.168504
4,CCD++ (Spark),0.927611,2777.976382
